In [1]:
import logging
from functools import partial
from typing import Tuple, List

import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, RandomSampler, DataLoader
from torchmetrics.classification import (
    MultilabelAccuracy,
    MultilabelPrecision,
    MultilabelRecall,
    MultilabelF1Score,
    MultilabelAUROC,
)
from tqdm import tqdm
from transformers import (
    BertTokenizer,
    BertModel,
    AdamW,
    get_linear_schedule_with_warmup,
)

import wandb


In [2]:
%env WANDB_SILENT=True
%env CUDA_LAUNCH_BLOCKING=1

logger = logging.getLogger("wandb")
logger.setLevel(logging.ERROR)

env: WANDB_SILENT=True
env: CUDA_LAUNCH_BLOCKING=1


In [3]:
bert_model_name = "bert-base-cased"
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda:0")

tokenizer = BertTokenizer.from_pretrained(bert_model_name)
assert (
        tokenizer.pad_token_id == 0
), "Padding value used in masks is set to zero, please change it everywhere"

train_df = pd.read_csv("data/train.csv")
# train_df = train_df.sample(frac=0.33)
train_df, val_df = train_test_split(train_df, test_size=0.05)

In [4]:
class ToxicDataset(Dataset):
    def __init__(
            self,
            tokenizer: BertTokenizer,
            dataframe: pd.DataFrame,
            lazy: bool = False,
    ):
        self.tokenizer = tokenizer
        self.pad_idx = tokenizer.pad_token_id
        self.lazy = lazy

        if not self.lazy:
            self.X = []
            self.Y = []
            for i, (row) in tqdm(dataframe.iterrows()):
                x, y = self.row_to_tensor(self.tokenizer, row)
                self.X.append(x)
                self.Y.append(y)
        else:
            self.df = dataframe

    @staticmethod
    def row_to_tensor(
            tokenizer: BertTokenizer,
            row: pd.Series,
    ) -> Tuple[torch.LongTensor, torch.LongTensor]:
        tokens = tokenizer.encode(row["comment_text"], add_special_tokens=True)
        if len(tokens) > 120:
            tokens = tokens[:119] + [tokens[-1]]

        x = torch.LongTensor(tokens)
        y = torch.FloatTensor(
                row[
                    [
                        "toxic",
                        "severe_toxic",
                        "obscene",
                        "threat",
                        "insult",
                        "identity_hate",
                    ]
                ]
        )
        return x, y

    def __len__(self):
        if self.lazy:
            return len(self.df)
        else:
            return len(self.X)

    def __getitem__(
            self,
            index: int,
    ) -> Tuple[torch.LongTensor, torch.LongTensor]:
        if not self.lazy:
            return self.X[index], self.Y[index]
        else:
            return self.row_to_tensor(self.tokenizer, self.df.iloc[index])


def collate_fn(
        batch: List[Tuple[torch.LongTensor, torch.LongTensor]],
        device: torch.device,
) -> Tuple[torch.LongTensor, torch.LongTensor]:
    x, y = list(zip(*batch))
    x = pad_sequence(x, batch_first=True, padding_value=0)
    y = torch.stack(y)
    return x.to(device), y.to(device)


train_dataset = ToxicDataset(tokenizer, train_df, lazy=True)
dev_dataset = ToxicDataset(tokenizer, val_df, lazy=True)

collate_fn = partial(collate_fn, device=device)

BATCH_SIZE = 128

train_sampler = RandomSampler(train_dataset)
dev_sampler = RandomSampler(dev_dataset)

train_iterator = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        collate_fn=collate_fn,
)
dev_iterator = DataLoader(
        dev_dataset,
        batch_size=BATCH_SIZE,
        sampler=dev_sampler,
        collate_fn=collate_fn,
)

In [5]:
class BertClassifier(nn.Module):
    def __init__(
            self,
            bert: BertModel,
            num_classes: int,
    ):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Linear(bert.config.hidden_size, num_classes)

    def forward(
            self,
            input_ids,
            attention_mask=None,
            token_type_ids=None,
            position_ids=None,
            head_mask=None,
            labels=None,
    ):
        outputs = self.bert(
                input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                position_ids=position_ids,
                head_mask=head_mask,
        )
        cls_output = outputs[1]  # batch, hidden
        cls_output = self.classifier(cls_output)  # batch, 6
        cls_output = torch.sigmoid(cls_output)

        criterion = nn.BCELoss()
        loss = 0
        if labels is not None:
            loss = criterion(cls_output, labels)

        return loss, cls_output


model = BertClassifier(BertModel.from_pretrained(bert_model_name), 6).to(device)

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [6]:
num_classes = 6
average_mode = "micro"
threshold = 0.5


In [7]:
def train(model, iterator, optimizer, scheduler):
    model.train()

    total_loss = 0
    accuracy = MultilabelAccuracy(
            num_labels=num_classes,
            average=average_mode,
            threshold=threshold,
    ).to(device=device)
    precision = MultilabelPrecision(
            num_labels=num_classes,
            average=average_mode,
            threshold=threshold,
    ).to(device=device)
    recall = MultilabelRecall(
            num_labels=num_classes,
            average=average_mode,
            threshold=threshold,
    ).to(device=device)
    f1_score = MultilabelF1Score(
            num_labels=num_classes,
            average=average_mode,
            threshold=threshold,
    ).to(device=device)
    auroc = MultilabelAUROC(
            num_labels=num_classes,
            average=average_mode,
    ).to(device=device)

    for text_batch, label_batch in tqdm(iterator):
        # Generate Predictions
        mask = (text_batch != 0).float()
        loss, pred = model(text_batch, attention_mask=mask, labels=label_batch)

        # Calculate loss
        total_loss += loss.item()

        accuracy.update(pred, label_batch)
        precision.update(pred, label_batch)
        recall.update(pred, label_batch)
        f1_score.update(pred, label_batch)
        auroc.update(pred, label_batch.to(torch.long))

        # Compute gradients
        loss.backward()

        # Update parameters using gradients
        optimizer.step()
        scheduler.step()

        # Reset the gradients to zero
        optimizer.zero_grad()

    return {
        "Training loss": total_loss / len(iterator),
        "Training accuracy": accuracy.compute().item(),
        "Training precision": precision.compute().item(),
        "Training recall": recall.compute().item(),
        "Training F1_score": f1_score.compute().item(),
        "Training AUROC": auroc.compute().item(),

    }


def evaluate(model, iterator):
    model.eval()

    accuracy = MultilabelAccuracy(
            num_labels=num_classes,
            average=average_mode,
            threshold=threshold,
    ).to(device=device)
    precision = MultilabelPrecision(
            num_labels=num_classes,
            average=average_mode,
            threshold=threshold,
    ).to(device=device)
    recall = MultilabelRecall(
            num_labels=num_classes,
            average=average_mode,
            threshold=threshold,
    ).to(device=device)
    f1_score = MultilabelF1Score(
            num_labels=num_classes,
            average=average_mode,
            threshold=threshold,
    ).to(device=device)
    auroc = MultilabelAUROC(
            num_labels=num_classes,
            average=average_mode,
    ).to(device=device)

    with torch.no_grad():
        total_loss = 0
        for text_batch, label_batch in tqdm(iterator):
            mask = (text_batch != 0).float()
            loss, pred = model(text_batch, attention_mask=mask, labels=label_batch)
            total_loss += loss

            accuracy.update(pred, label_batch)
            precision.update(pred, label_batch)
            recall.update(pred, label_batch)
            f1_score.update(pred, label_batch)
            auroc.update(pred, label_batch.to(torch.long))

    return {
        "Validation loss": total_loss / len(iterator),
        "Validation accuracy": accuracy.compute().item(),
        "Validation precision": precision.compute().item(),
        "Validation recall": recall.compute().item(),
        "Validation F1_score": f1_score.compute().item(),
        "Validation AUROC": auroc.compute().item(),
    }


In [8]:
no_decay = ["bias", "LayerNorm.weight"]

optimizer_grouped_parameters = [
    {
        "params": [
            p
            for n, p in model.named_parameters()
            if not any(nd in n for nd in no_decay)
        ],
        "weight_decay": 0.01,
    },
    {
        "params": [
            p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)
        ],
        "weight_decay": 0.0,
    },
]

EPOCH_NUM = 2
warmup_steps = 10 ** 3

total_steps = len(train_iterator) * EPOCH_NUM - warmup_steps
optimizer = AdamW(optimizer_grouped_parameters, lr=2e-5, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

/home/sigma/projects/PycharmProjects/toxic-comment-detection/venv/lib/python3.11/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [9]:
wandb.init(
        project="Toxic Comment Detection",
        name="BERT",
        config={
            "learning_rate": 2e-5,
            "architecture": "BERT",
            "optimizer": "adamW",
            "epochs": 2,
        },
        anonymous="allow",
)

In [10]:
log_values = {}

In [11]:
from pprint import pprint

for i in range(EPOCH_NUM):
    train_logs = train(model, train_iterator, optimizer, scheduler)
    val_logs = evaluate(model, dev_iterator)

    train_logs.update(val_logs)

    wandb.log(train_logs)
    pprint(train_logs)

    train_logs.update({"Epoch": int(i + 1)})

    for key in train_logs.keys():
        train_logs[key] = float(format(train_logs[key], ".3f"))

    log_values[i + 1] = train_logs

100%|██████████| 63/63 [00:35<00:00,  1.76it/s]


{'Training AUROC': 0.9231317043304443,
 'Training F1_score': 0.5594099760055542,
 'Training accuracy': 0.9725469350814819,
 'Training loss': 0.12453474836730505,
 'Training precision': 0.6772332787513733,
 'Training recall': 0.4765082597732544,
 'Validation AUROC': 0.9886984825134277,
 'Validation F1_score': 0.7590290307998657,
 'Validation accuracy': 0.9829970598220825,
 'Validation loss': tensor(0.0473, device='cuda:0'),
 'Validation precision': 0.8287007212638855,
 'Validation recall': 0.7001638412475586}


100%|██████████| 63/63 [00:36<00:00,  1.75it/s]

{'Training AUROC': 0.9902490973472595,
 'Training F1_score': 0.7849065065383911,
 'Training accuracy': 0.9852784872055054,
 'Training loss': 0.041957081599703315,
 'Training precision': 0.8428842425346375,
 'Training recall': 0.734391450881958,
 'Validation AUROC': 0.9895849227905273,
 'Validation F1_score': 0.7722007632255554,
 'Validation accuracy': 0.9827463626861572,
 'Validation loss': tensor(0.0473, device='cuda:0'),
 'Validation precision': 0.7799443006515503,
 'Validation recall': 0.7646095156669617}


In [12]:
import collections

all_series = collections.deque()
for (key, log_dict) in log_values.items():
    all_series.append(pd.Series(log_dict))

df = pd.DataFrame(all_series)
df.to_csv(f"metric_logs/BERT.csv", index=False)

In [13]:
torch.save(model.state_dict(), "model_weights/BERT.pth")

In [ ]:
df_log=pd.read_csv("metric_logs/BERT.csv",)
df_log.head()